# Fine-tuned DeBERTa (step 9) on Google Colab

Fine-tunes `microsoft/deberta-v3-small` on the scam-call data (phone calls only, trained on synthetic calls — Option 1)
and evaluates it on the same splits as the other models: Test A, Test B (unseen sources + real calls) and Test C (first 50 words).

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Cell 2 asks you to upload the **new `colab_bundle.zip`** (it contains the DeBERTa code).
Training runs 6 times (one per split) — about 25–40 minutes in total.
The last cell downloads **`deberta_results.zip`** — put it in the project folder.

In [ ]:
# 1. Check the GPU (should say Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload the NEW colab_bundle.zip and unpack it (replaces any older copy)
import os, shutil, zipfile
from google.colab import files

%cd /content
for old in ('colab_bundle.zip', 'scam-classifier'):
    if os.path.isdir(old): shutil.rmtree(old)
    elif os.path.exists(old): os.remove(old)
uploaded = files.upload()  # pick colab_bundle.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall('scam-classifier')
%cd /content/scam-classifier
!grep -c "class DebertaClassifier" src/models.py  # must print 1

In [ ]:
# 3. Packages (Colab already has torch, transformers, pandas, scikit-learn)
!pip -q install sentencepiece protobuf tabulate
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
# 4. Fine-tune + evaluate DeBERTa on every split (A, B_*, plus first-50-words variants)
EPOCHS = 3
!python src/run_experiment.py --use-transformer --deberta-epochs {EPOCHS} 2>&1 | grep -vE "Warning|warn"

In [ ]:
# 5. Show the results
from IPython.display import Markdown, display
display(Markdown(open('results/summary_deberta.md').read()))

In [ ]:
# 6. Download the results
!zip -q -r deberta_results.zip results/metrics_deberta*.csv results/summary_deberta.md
files.download('deberta_results.zip')